# 08 · Does connectivity reconfiguration really add something?

**Owner:** Jaime
**Status:** sandbox, for team discussion (not a decision)
**Dataset:** B (`load_hcp`, 336 subjects)
**Date:** 2026-07-21

### What this notebook is for

If you are picking this up to run your own analysis, read the cells in order. Each step builds on the previous one and says what it found before moving on.

The submitted abstract says that load reconfiguration gives *"complementary information beyond static FC."* We wrote that before testing it directly. This notebook tests it.

Two things came out of that test:

1. Reconfiguration does predict working-memory accuracy, but it does **not** clearly beat plain single-condition connectivity once you compare them fairly.
2. A much simpler feature, **how strongly each brain region activates during the task**, predicts accuracy a lot better than any connectivity feature.

So the notebook has one job: check, carefully, whether that second result is **real or just an artifact**, and hand the team the evidence to decide what to do with it.

### How to trust the numbers

Everything reuses the team pipeline exactly as it is in notebook `04`:

- Goutham's `connect_dots` and `get_brain_profile` functions
- the 78-value network fingerprint
- the 4-second HRF delay
- the seed-42 `RidgeCV` model

Before any new claim, a **reproduction gate** recomputes the canonical reconfiguration number (r about 0.366) and stops the notebook if it does not match. If the gate passes, this is the same pipeline the team already agreed on.

### Results at a glance

| Question | Answer |
|---|---|
| Does reconfiguration predict working memory? | Yes. r about **0.366**, external B to A about 0.40 (notebooks `04` / `05`). This holds. |
| Does reconfiguration add over 0-back FC? | Not clearly. delta R2 = **+0.034 (sd 0.023)**, under 2 sd. Combining `0-back + reconfig` (0.333) is even *worse* than reconfig alone. |
| Does task activation (2-back vs 0-back) predict? | Yes, much better. r about **0.60**. Survives motion, ability, permutation, and cross-run. |
| Does connectivity add over activation? | No. delta R2 about **-0.003**. |
| Is this "reconfiguration of activation"? | No. 0-back activation *alone* already predicts about 0.57. The signal is a stable individual trait, not a load change. |
| Why does activation win? | It is about **7x more reliable** between runs (0.17 to 0.25) than the difference-of-correlations fingerprint (0.02). |

**In one line:** connectivity reconfiguration is a real but weak and non-specific predictor. What actually drives the prediction is how strongly the cortex activates during the task, which is a very stable trait of each person.

One caveat applies to *everything* below equally, both connectivity and activation: same-task circularity. It is spelled out at the end.

## Step 0 · Setup

We load the shared A/B data layer and paste Goutham's connectivity functions **exactly** as they are in notebook `04`. Nothing here is new.

Using the same functions is what lets us reproduce the canonical number later and trust the comparison.

In [1]:
from pathlib import Path
import os, sys, time
import numpy as np
from scipy.stats import pearsonr
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

HERE = Path.cwd(); JAIME = HERE if (HERE/"datasets.py").exists() else HERE/"sandbox"/"jaime"
sys.path.insert(0, str(JAIME))
import datasets as ds, preprocessing as pp
DATA = Path(os.environ.get("GAMMAS_DATA_DIR", JAIME.parents[1]/"data"))
B = ds.spec_b(DATA); DELAY = 4.0
net = np.load(B.task_dir/"regions.npy").T[1]; soc = np.unique(net)

# ----- Goutham's functions, verbatim (guarantees the fingerprint matches nb04) -----
def connect_dots(x): return np.corrcoef(x)
def get_brain_profile(bs, nn, ss):
    fp = []
    for i, ga in enumerate(ss):
        ia = np.where(nn == ga)[0]
        for gb in ss[i:]:
            ib = np.where(nn == gb)[0]
            if len(ia) == 0 or len(ib) == 0: fp.append(0.0); continue
            if ga == gb:
                sub = bs[np.ix_(ia, ia)]; n = sub.shape[0]
                fp.append(sub[np.triu_indices(n, 1)].mean() if n > 1 else 0.0)
            else: fp.append(bs[np.ix_(ia, ib)].mean())
    return fp

crystal = lambda: RidgeCV(alphas=np.logspace(-3, 5, 100))        # team model
def cvp(X, y, seed=42):                                          # leakage-free 5-fold OOF prediction
    pred = np.zeros_like(y, float)
    for tr, te in KFold(5, shuffle=True, random_state=seed).split(X):
        sc = StandardScaler().fit(X[tr]); m = crystal().fit(sc.transform(X[tr]), y[tr])
        pred[te] = m.predict(sc.transform(X[te]))
    return pred
def rep(X, y, seeds=range(20)):                                 # repeated CV -> mean, sd of r
    r = [pearsonr(cvp(X, y, s), y)[0] for s in seeds]; return np.mean(r), np.std(r)
print("ready:", B.name)

ready: Finalist B (339 subj, +resting-state)


## Step 1 · Build the features

One pass over dataset B, two scans (runs) per subject. From the same task frames (shifted 4 seconds forward to account for the hemodynamic delay) we pull two kinds of feature:

- **Connectivity fingerprint** (78 values): the network-level correlation pattern, computed for 0-back, for 2-back, and for the reconfiguration (2-back minus 0-back).
- **Activation** (360 values): the average BOLD signal of each region, per condition.

Two versions of each are kept:

- **pooled**: both runs together, used for the main models.
- **per-run**: each run on its own, used later for the cross-run test.

`DVARS` is a simple head-motion proxy computed from the BOLD signal. We use it as a control.

In [2]:
A0 = {0: [], 1: []}; A2 = {0: [], 1: []}; FPr = {0: [], 1: []}
A0p, A2p, FP0p, FPp, dvars = [], [], [], [], []
subs = ds.load_subjects(B); t0 = time.time()
for k, s in enumerate(subs):
    ts = {r: ds.load_timeseries(B, s, r) for r in (0, 1)}
    fr = {(r, lvl): (lambda f: f[f < ts[r].shape[1]])(pp.condition_frames(B, s, r, lvl, DELAY))
          for r in (0, 1) for lvl in ("0back", "2back")}
    for r in (0, 1):
        A0[r].append(ts[r][:, fr[(r, "0back")]].mean(1)); A2[r].append(ts[r][:, fr[(r, "2back")]].mean(1))
        FPr[r].append(get_brain_profile(connect_dots(ts[r][:, fr[(r, "2back")]]) -
                                        connect_dots(ts[r][:, fr[(r, "0back")]]), net, soc))
    t0f = np.concatenate([ts[0][:, fr[(0, "0back")]], ts[1][:, fr[(1, "0back")]]], 1)
    t2f = np.concatenate([ts[0][:, fr[(0, "2back")]], ts[1][:, fr[(1, "2back")]]], 1)
    A0p.append(t0f.mean(1)); A2p.append(t2f.mean(1))
    FP0p.append(get_brain_profile(connect_dots(t0f), net, soc))                     # 0-back FC fingerprint
    FPp.append(get_brain_profile(connect_dots(t2f) - connect_dots(t0f), net, soc))  # reconfiguration
    dvars.append(np.mean([np.sqrt((np.diff(ts[r], axis=1)**2).mean(0)).mean() for r in (0, 1)]))
A0 = {r: np.array(v) for r, v in A0.items()}; A2 = {r: np.array(v) for r, v in A2.items()}
FPr = {r: np.array(v) for r, v in FPr.items()}
A0p, A2p, FP0p, FPp, dvars = map(np.array, (A0p, A2p, FP0p, FPp, dvars))
beh = pp.behaviour_table(B).set_index("subject").loc[subs]
y = beh["acc_2bk"].to_numpy(float); acc0 = beh["acc_0bk"].to_numpy(float)
act_c = A2p - A0p
print(f"{len(y)} subjects | FC {FPp.shape[1]}-dim | activation {A0p.shape[1]}-dim | {time.time()-t0:.0f}s")

336 subjects | FC 78-dim | activation 360-dim | 7s


## Step 2 · Reproduction gate

Before trusting anything new, recompute the number the team already reported (reconfiguration r about 0.366). If it does not match, the notebook stops here.

This guarantees we are standing on the same pipeline as notebook `04`.

In [3]:
g = rep(FPp, y)[0]
print(f"reconfiguration FC repeated-CV r = {g:+.3f}  (nb04 canonical: +0.366)")
assert abs(g - 0.366) < 0.03, "pipeline does not reproduce nb04 -- stop"
print("GATE PASSED")

reconfiguration FC repeated-CV r = +0.366  (nb04 canonical: +0.366)
GATE PASSED


## Step 3 · Model panel

Same setup for every feature: repeated 5-fold cross-validation, 20 seeds, predicting 2-back accuracy (`acc_2bk`). We only swap the input features and read off the cross-validated correlation.

Watch two things in the output:

- activation (about 0.60) predicts much better than connectivity (about 0.37).
- 0-back activation *alone* (about 0.57) is almost as good as the 2-back-minus-0-back contrast. That is the first hint this is not about task load.

In [4]:
panel = [("reconfig FC (78)", FPp), ("activation contrast 2-0 (360)", act_c),
         ("activation 2-back alone (360)", A2p), ("activation 0-back alone (360)", A0p)]
print(f"{'model':32s} {'CV r':>16s}")
for nm, X in panel:
    m, sd = rep(X, y); print(f"{nm:32s} {m:+.3f} +/- {sd:.3f}")
print("\n-> activation predicts ~0.60 vs FC 0.37; and 0-back activation ALONE ~0.57,")
print("   so the signal is trait-level activation, NOT the load contrast.")

model                                        CV r


reconfig FC (78)                 +0.366 +/- 0.024


activation contrast 2-0 (360)    +0.600 +/- 0.016


activation 2-back alone (360)    +0.569 +/- 0.015


activation 0-back alone (360)    +0.571 +/- 0.014

-> activation predicts ~0.60 vs FC 0.37; and 0-back activation ALONE ~0.57,
   so the signal is trait-level activation, NOT the load contrast.


## Step 4 · Does the fancier feature actually add anything?

A higher correlation on its own is not proof. The fair test is nested and uses the **same folds**: fit the simple feature first, then check whether adding the fancier one improves out-of-sample R2.

We ask it twice:

- does reconfiguration add over plain 0-back connectivity?
- does connectivity add over activation?

In [5]:
def delta(full, base):                     # incremental R2 on identical folds, per seed (paired)
    d = [r2_score(y, cvp(full, y, s)) - r2_score(y, cvp(base, y, s)) for s in range(20)]
    return np.mean(d), np.std(d)
di, si = delta(np.hstack([FP0p, FPp]), FP0p)   # does reconfiguration add over 0-back FC?
d1, s1 = delta(np.hstack([FPp, act_c]), act_c) # does FC add over activation?
print(f"reconfig FC over 0-back FC : Delta R2 = {di:+.4f} +/- {si:.4f}  [{'adds' if di-2*si>0 else 'no clear gain'}]")
print(f"FC over activation         : Delta R2 = {d1:+.4f} +/- {s1:.4f}  [{'adds' if d1-2*s1>0 else 'no clear gain'}]")

reconfig FC over 0-back FC : Delta R2 = +0.0344 +/- 0.0225  [no clear gain]
FC over activation         : Delta R2 = -0.0030 +/- 0.0065  [no clear gain]


## Step 5 · Is the activation result real? Controls

The activation number is big, so we try to break it. Each control removes one alternative explanation:

- **General ability** (`acc_0bk`): maybe "smart people activate more"? Partial it out.
- **Head motion** (`DVARS`): maybe a movement artifact? Partial it out, and check the direct correlation.
- **Permutation**: shuffle the labels 1000 times and see how often chance beats the real result.

If the number survives all three, it is not one of these artifacts.

In [6]:
def resid(v, z):
    Zc = np.column_stack([np.ones(len(v)), z]); return v - Zc @ np.linalg.lstsq(Zc, v, rcond=None)[0]
def partial(pred, z): return pearsonr(resid(pred, z), resid(y, z))[0]
def permp(X, obs, n=1000):
    rng = np.random.default_rng(0)
    return (sum(pearsonr(cvp(X, rng.permutation(y)), y)[0] >= obs for _ in range(n)) + 1)/(n+1)
pred = cvp(act_c, y); r = pearsonr(pred, y)[0]
print(f"activation contrast   raw r        = {r:+.3f}")
print(f"  partial | acc_0bk (ability)      = {partial(pred, acc0):+.3f}")
print(f"  partial | DVARS (motion)         = {partial(pred, dvars):+.3f}")
print(f"  partial | acc_0bk + DVARS        = {partial(pred, np.column_stack([acc0, dvars])):+.3f}")
print(f"  permutation p (1000)             = {permp(act_c, r):.4f}")
print(f"  corr(prediction, DVARS)          = {pearsonr(pred, dvars)[0]:+.3f}  (weak -> not motion)")

activation contrast   raw r        = +0.598
  partial | acc_0bk (ability)      = +0.412
  partial | DVARS (motion)         = +0.580
  partial | acc_0bk + DVARS        = +0.411


  permutation p (1000)             = 0.0010
  corr(prediction, DVARS)          = -0.190  (weak -> not motion)


## Step 6 · The decisive test: predict across runs

This is the strongest check. Train on **one** scan's features (all subjects), then predict the **other** scan.

The two scans are separate recordings, so nothing from the test scan leaks into training. If the earlier result were just within-run reliability inflating the score, cross-run prediction would collapse.

It does not collapse: activation stays around 0.53, while connectivity sits near 0.28.

In [7]:
def cross_run(F0, F1):
    sc = StandardScaler().fit(F0); ra = pearsonr(crystal().fit(sc.transform(F0), y).predict(sc.transform(F1)), y)[0]
    sc = StandardScaler().fit(F1); rb = pearsonr(crystal().fit(sc.transform(F1), y).predict(sc.transform(F0)), y)[0]
    return (ra + rb) / 2
print(f"activation contrast cross-run r = {cross_run(A2[0]-A0[0], A2[1]-A0[1]):+.3f}")
print(f"reconfig FC         cross-run r = {cross_run(FPr[0], FPr[1]):+.3f}   (matches nb04 §3b)")

activation contrast cross-run r = +0.526
reconfig FC         cross-run r = +0.280   (matches nb04 §3b)


## Step 7 · Why activation wins: reliability

The reconfiguration fingerprint is a **difference between two correlation matrices**. The two conditions are very similar, so subtracting them mostly leaves noise. Its between-run reliability is about 0.02, essentially zero.

Mean activation is a much steadier per-region measurement, so its reliability is far higher.

A feature you can barely measure the same way twice cannot predict well, which is exactly what the next cell shows.

In [8]:
def rel(F0, F1):
    rr = [pearsonr(F0[:, j], F1[:, j])[0] for j in range(F0.shape[1])]
    return float(np.tanh(np.nanmean(np.arctanh(np.clip(rr, -0.999, 0.999)))))
print(f"activation contrast : {rel(A2[0]-A0[0], A2[1]-A0[1]):+.3f}")
print(f"activation 2-back   : {rel(A2[0], A2[1]):+.3f}")
print(f"reconfig FC         : {rel(FPr[0], FPr[1]):+.3f}")

activation contrast : +0.169
activation 2-back   : +0.249


reconfig FC         : +0.024


## What to take from this (for team discussion, not a decision)

**Solid and checked:**

1. Connectivity reconfiguration predicts working-memory accuracy (r about 0.37, replicated; external B to A about 0.40). This stands.
2. But it does not clearly add over single-condition connectivity (nested delta R2 about +0.03, under 2 sd; `0-back + reconfig` is no better than reconfig alone).
3. Task activation predicts much better (r about 0.60), and it is not an artifact:
   - survives motion (partial | DVARS about 0.58)
   - survives general ability (partial | acc_0bk about 0.41)
   - permutation p about 0.001
   - holds across independent runs (cross-run r about 0.53)
4. Connectivity adds nothing on top of activation (delta R2 about -0.003).
5. It is a stable trait, not a load effect: 0-back activation alone predicts about 0.57, almost as well as the contrast. So this is not an "activation reconfiguration" story.
6. The reason is reliability: activation is about 7x more reproducible between runs than the reconfiguration fingerprint.

**What this does not mean.** It does not invalidate the project. Connectivity prediction is real and externally validated. It changes *the explanation*: the strongest and most reliable predictor of working memory here is how strongly the cortex activates during the task, an individual trait that is not specific to connectivity and not specific to load.

**The caveat that applies to all of it.** Every analysis here, connectivity and activation alike, predicts N-back accuracy from features derived from the N-back task itself, so they all share task-state variance. The `acc_0bk` and cross-run controls make the activation result non-trivial, but it is still an associational, same-task result, not a causal one.

**Next step (21 Jul).** Waiting on Goutham's pipeline. His approach may already cover this (for example if his main result is activation-based), or the team may keep the connectivity framing and cite this as an honest limitation. That call belongs to the group. This notebook is the evidence for it.